# Real-Time Resume Ranking

## Objective

This notebook simulates a real-world recruitment workflow.

The system will:

1. Accept a job description
2. Read uploaded PDF resumes
3. Extract resume text
4. Generate embeddings
5. Rank candidates
6. Provide explanations

This notebook serves as the prototype for the Streamlit application.

In [1]:
import fitz
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\KIIT\OneDrive\Desktop\TalentMatch_AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Transformer Model

In [2]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4422.11it/s]


In [4]:
job_description = """
Financial Analyst

Requirements:
Financial Modeling
Budget Forecasting
Risk Analysis
Accounting
Microsoft Excel
Data Analysis
Financial Reporting

Education:
B.Com / MBA Finance

Experience:
2+ years
"""

In [5]:
job_description

'\nFinancial Analyst\n\nRequirements:\nFinancial Modeling\nBudget Forecasting\nRisk Analysis\nAccounting\nMicrosoft Excel\nData Analysis\nFinancial Reporting\n\nEducation:\nB.Com / MBA Finance\n\nExperience:\n2+ years\n'

In [6]:
jd_profile = {
    "Required_Skills": [
        "Financial Modeling",
        "Budget Forecasting",
        "Risk Analysis",
        "Accounting",
        "Microsoft Excel",
        "Data Analysis",
        "Financial Reporting"
    ],
    "Education": [
        "B.Com",
        "MBA Finance"
    ],
    "Experience": [
        "2+ years"
    ]
}

jd_profile

{'Required_Skills': ['Financial Modeling',
  'Budget Forecasting',
  'Risk Analysis',
  'Accounting',
  'Microsoft Excel',
  'Data Analysis',
  'Financial Reporting'],
 'Education': ['B.Com', 'MBA Finance'],
 'Experience': ['2+ years']}

# Load Resume Dataset

In [7]:

import pandas as pd

df = pd.read_csv("../data/processed/final_clean_resume_dataset.csv")

print(df.shape)
df.head()

(2458, 4)


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


# Create Resume Embeddings

In [12]:
print(df.columns.tolist())

['ID', 'Resume_str', 'Resume_html', 'Category']


In [14]:
resume_embeddings = model.encode(
    df["Resume_str"].tolist(),
    show_progress_bar=True
)

print(resume_embeddings.shape)

Batches:   0%|          | 0/77 [00:00<?, ?it/s]

Batches: 100%|██████████| 77/77 [00:57<00:00,  1.35it/s]

(2458, 384)


# Create Job Description Text

In [15]:
jd_text = " ".join(jd_profile["Required_Skills"]) + " " + \
          " ".join(jd_profile["Education"]) + " " + \
          " ".join(jd_profile["Experience"])

jd_text

'Financial Modeling Budget Forecasting Risk Analysis Accounting Microsoft Excel Data Analysis Financial Reporting B.Com MBA Finance 2+ years'

# Generate Job Description Embedding

In [16]:
jd_embedding = model.encode([jd_text])

jd_embedding.shape

(1, 384)

# Calculate Similarity Scores

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    jd_embedding,
    resume_embeddings
)

similarities.shape

(1, 2458)

# Rank Candidates

In [18]:
df["Similarity_Score"] = similarities[0] * 100

ranked_df = df.sort_values(
    by="Similarity_Score",
    ascending=False
)

ranked_df[
    ["ID", "Category", "Similarity_Score"]
].head(10)

,ID,Category,Similarity_Score
1566,59450123,FINANCE,62.162781
948,16507693,AGRICULTURE,59.649456
1500,81677620,FINANCE,57.884895
1503,11441764,FINANCE,57.717525
1862,49997097,ACCOUNTANT,56.308842
2206,18824120,BANKING,55.858814
1506,62850928,FINANCE,55.762733
1904,82649935,ACCOUNTANT,55.089111
1861,27637576,ACCOUNTANT,54.923470
1901,50222417,ACCOUNTANT,54.388027


# View Top Candidates

In [19]:
top_candidates = ranked_df.head(10)

top_candidates[
    [
        "ID",
        "Category",
        "Similarity_Score"
    ]
]

,ID,Category,Similarity_Score
1566,59450123,FINANCE,62.162781
948,16507693,AGRICULTURE,59.649456
1500,81677620,FINANCE,57.884895
1503,11441764,FINANCE,57.717525
1862,49997097,ACCOUNTANT,56.308842
2206,18824120,BANKING,55.858814
1506,62850928,FINANCE,55.762733
1904,82649935,ACCOUNTANT,55.089111
1861,27637576,ACCOUNTANT,54.923470
1901,50222417,ACCOUNTANT,54.388027


# Save Ranked Candidates

In [20]:
top_candidates.to_csv(
    "../data/processed/realtime_ranked_candidates.csv",
    index=False
)

print("Ranking saved successfully.")

Ranking saved successfully.
